# Embedding + Chroma 最小实验

这个 Notebook 只使用 3 条短文本，观察 `文本 → 向量 → Chroma → 问题向量 → Top-K`。完整知识库请使用 `run.py`。

In [2]:
from pathlib import Path
import sys
import chromadb

module_root = Path.cwd()
if module_root.name != "06-embedding-basics":
    module_root = module_root / "06-embedding-basics"
if not module_root.exists():
    raise FileNotFoundError(f"找不到模块目录：{module_root}")
sys.path.insert(0, str(module_root))

# 导入 embedding_lab 时会先加载 studyAgent 根目录的 .env。
from embedding_lab.config import Settings
from embedding_lab.embeddings import OpenAICompatibleEmbeddings

settings = Settings()
settings.require_embedding_config()
embeddings = OpenAICompatibleEmbeddings(
    api_key=settings.embedding_api_key,
    base_url=settings.embedding_base_url,
    model=settings.embedding_model,
)
print("Embedding 模型：", settings.embedding_model)

Embedding 模型： qwen3.7-text-embedding-flash


In [3]:
texts = [
    "RAG 会先检索知识，再让聊天模型回答。",
    "Chroma 是可以持久化到本地的向量数据库。",
    "玻璃幕墙中的窗需要记录它所依附的父墙。",
]
document_vectors = embeddings.embed_documents(texts)

print("文本数量：", len(texts))
print("向量数量：", len(document_vectors))
print("向量维度：", len(document_vectors[0]))
print("第一个向量的前 5 项：", document_vectors[0][:5])

文本数量： 3
向量数量： 3
向量维度： 1024
第一个向量的前 5 项： [-0.00913238525390625, -0.01277923583984375, -0.0258636474609375, -0.007770538330078125, -0.01152801513671875]


In [7]:
client = chromadb.PersistentClient(path=str(settings.persist_dir))
collection = client.get_or_create_collection(
    name="embedding_three_text_demo_v1",
    metadata={"hnsw:space": "cosine"},
)

# upsert 可以安全重复运行；add 再次写相同 ID 容易报重复错误。
collection.upsert(
    ids=["demo-1", "demo-2", "demo-3"],
    documents=texts,
    embeddings=document_vectors,
    metadatas=[
        {"topic": "RAG"},
        {"topic": "Chroma"},
        {"topic": "building"},
    ],
)
print("集合中文本块数量：", collection.count())

集合中文本块数量： 3


In [8]:
question = "幕墙的窗和墙是什么关系？"
query_vector = embeddings.embed_query(question)
result = collection.query(
    query_embeddings=[query_vector],
    n_results=3,
    include=["documents", "metadatas", "distances"],
)

for rank, (document, metadata, distance) in enumerate(
    zip(result["documents"][0], result["metadatas"][0], result["distances"][0]),
    start=1,
):
    print(f"Top {rank} | distance={distance:.4f} | {metadata}")
    print(document)

Top 1 | distance=0.3109 | {'topic': 'building'}
玻璃幕墙中的窗需要记录它所依附的父墙。
Top 2 | distance=0.6983 | {'topic': 'RAG'}
RAG 会先检索知识，再让聊天模型回答。
Top 3 | distance=0.8045 | {'topic': 'Chroma'}
Chroma 是可以持久化到本地的向量数据库。


## 你应该观察什么

- 文档在建库时调用一次 Embedding；问题在查询时也调用一次。
- Chroma 返回原文、metadata 和距离，而不是自然语言答案。
- 在 cosine 距离下，本例通常是距离越小越相关。
- 关闭 Notebook 再打开，集合仍在磁盘中；持久化不等于每次重算向量。